In [5]:
pip install opencv-python


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import cv2
print("OpenCV version:", cv2.__version__)

OpenCV version: 4.13.0


In [3]:
import cv2

def test_video(video_path):
    cap = cv2.VideoCapture(video_path)
    print(f"Testing: {video_path}")
    print("Opened:", cap.isOpened())
    
    ret, frame = cap.read()
    print("First frame read:", ret)
    
    if ret:
        print("Frame shape:", frame.shape)
    cap.release()
    print("-" * 30)

test_video("data/raw_videos/real/000.mp4")
test_video("data/raw_videos/fake/002_006.mp4")

Testing: data/raw_videos/real/000.mp4
Opened: True
First frame read: True
Frame shape: (480, 640, 3)
------------------------------
Testing: data/raw_videos/fake/002_006.mp4
Opened: True
First frame read: True
Frame shape: (720, 1280, 3)
------------------------------


In [4]:
import os
import cv2
import numpy as np

IMG_SIZE = 224

def load_images_from_folder(folder, label):
    images = []
    labels = []
    
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        
        if img is None:
            continue
        
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        images.append(img)
        labels.append(label)
    
    return images, labels

real_images, real_labels = load_images_from_folder("data/processed_frames/real", 0)
fake_images, fake_labels = load_images_from_folder("data/processed_frames/fake", 1)

X = np.array(real_images + fake_images)
y = np.array(real_labels + fake_labels)

print("Total samples:", len(X))
print("Labels:", y)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data/processed_frames/real'

In [5]:
import os
print(os.getcwd())


C:\Users\Arsh\OneDrive\Desktop\DeepScan-AI\DeepScan-AI


In [6]:
import os

os.makedirs("data/processed_frames/real", exist_ok=True)
os.makedirs("data/processed_frames/fake", exist_ok=True)

print("Folders created!")

Folders created!


In [7]:
import cv2
import os

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def extract_faces(video_path, save_dir, label, sample_rate=30, max_frames=20):
    cap = cv2.VideoCapture(video_path)
    count = 0
    saved = 0

    while True:
        ret, frame = cap.read()
        if not ret or saved >= max_frames:
            break

        if count % sample_rate == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.3, 5)

            for (x, y, w, h) in faces:
                face = frame[y:y+h, x:x+w]
                filename = f"{label}_{saved}.jpg"
                cv2.imwrite(os.path.join(save_dir, filename), face)
                saved += 1

        count += 1

    cap.release()
    print(f"Saved {saved} faces from {video_path}")

extract_faces("data/raw_videos/real/000.mp4", "data/processed_frames/real", "real")
extract_faces("data/raw_videos/fake/002_006.mp4", "data/processed_frames/fake", "fake")

Saved 14 faces from data/raw_videos/real/000.mp4
Saved 13 faces from data/raw_videos/fake/002_006.mp4


In [8]:
import os
print("Real faces:", os.listdir("data/processed_frames/real"))
print("Fake faces:", os.listdir("data/processed_frames/fake"))

Real faces: ['real_0.jpg', 'real_1.jpg', 'real_10.jpg', 'real_11.jpg', 'real_12.jpg', 'real_13.jpg', 'real_2.jpg', 'real_3.jpg', 'real_4.jpg', 'real_5.jpg', 'real_6.jpg', 'real_7.jpg', 'real_8.jpg', 'real_9.jpg']
Fake faces: ['fake_0.jpg', 'fake_1.jpg', 'fake_10.jpg', 'fake_11.jpg', 'fake_12.jpg', 'fake_2.jpg', 'fake_3.jpg', 'fake_4.jpg', 'fake_5.jpg', 'fake_6.jpg', 'fake_7.jpg', 'fake_8.jpg', 'fake_9.jpg']


In [11]:
print("Real images:", len(real_images))
print("Fake images:", len(fake_images))

NameError: name 'real_images' is not defined

In [12]:
import os
import cv2
import numpy as np

IMG_SIZE = 224

def load_images_from_folder(folder, label):
    images = []
    labels = []
    
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        
        if img is None:
            continue
        
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        images.append(img)
        labels.append(label)
    
    return images, labels

In [13]:
real_images, real_labels = load_images_from_folder("data/processed_frames/real", 0)
fake_images, fake_labels = load_images_from_folder("data/processed_frames/fake", 1)

print("Loaded real faces:", len(real_images))
print("Loaded fake faces:", len(fake_images))

Loaded real faces: 14
Loaded fake faces: 13


In [14]:
from sklearn.model_selection import train_test_split

X = np.array(real_images + fake_images)
y = np.array(real_labels + fake_labels)

X = X / 255.0  # normalize

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)

Train shape: (21, 224, 224, 3)
Val shape: (6, 224, 224, 3)


In [15]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(16, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

ModuleNotFoundError: No module named 'tensorflow'

In [16]:
!pip install tensorflow

   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\Arsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\Arsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ~~~~~~~~~~~~~^^^^^
  File "C:\Users\Arsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ~~~~~~~~~~~~~^^^^^
  File "C:\Users\Arsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ~~~~~~~~~~~~~~^^^^^
 

In [1]:
import tensorflow as tf
print(tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

In [1]:
import tensorflow as tf
print(tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

In [1]:
import tensorflow as tf
print(tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

In [1]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [19]:
import os
import cv2
import numpy as np

IMG_SIZE = 224

def load_images_from_folder(folder, label):
    images = []
    labels = []
    
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        
        if img is None:
            continue
        
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        images.append(img)
        labels.append(label)
    
    return images, labels

real_images, real_labels = load_images_from_folder("data/processed_frames/real", 0)
fake_images, fake_labels = load_images_from_folder("data/processed_frames/fake", 1)

print("Loaded real faces:", len(real_images))
print("Loaded fake faces:", len(fake_images))

Loaded real faces: 14
Loaded fake faces: 13


In [18]:
from sklearn.model_selection import train_test_split

X = np.array(real_images + fake_images)
y = np.array(real_labels + fake_labels)

X = X / 255.0

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (21, 224, 224, 3)
Val: (6, 224, 224, 3)


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(16, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\Arsh\OneDrive\Desktop\DeepScan-AI\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 93312)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     5,972,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,977,185 (22.80 MB)

 Trainable params: 5,977,185 (22.80 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
pred = model.predict(X[0:1])
print("Prediction score:", float(pred[0][0]))
print("0 = real, 1 = fake")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Prediction score: 0.44351089000701904
0 = real, 1 = fake


In [27]:
import os
import cv2
import numpy as np

IMG_SIZE = 224

def load_images_from_folder(folder, label):
    images = []
    labels = []
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        img = cv2.imread(path)
        if img is None:
            continue
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        images.append(img)
        labels.append(label)
    return images, labels

real_path = "data/processed_frames/real"
fake_path = "data/processed_frames/fake"

real_images, real_labels = load_images_from_folder(real_path, 0)
fake_images, fake_labels = load_images_from_folder(fake_path, 1)

X = np.array(real_images + fake_images) / 255.0
y = np.array(real_labels + fake_labels)

print("Real images:", len(real_images))
print("Fake images:", len(fake_images))
print("Total:", len(X))

Real images: 0
Fake images: 0
Total: 0


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", len(X_train))
print("Test:", len(X_test))

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(16, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\Arsh\OneDrive\Desktop\DeepScan-AI\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 222, 222, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 111, 111, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 109, 109, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 54, 54, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 93312)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │     5,972,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,977,185 (22.80 MB)

 Trainable params: 5,977,185 (22.80 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 453ms/step - accuracy: 0.6250 - loss: 2.5386 - val_accuracy: 1.0000 - val_loss: 0.0746
Epoch 2/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 1.0000 - loss: 0.2015 - val_accuracy: 1.0000 - val_loss: 0.0182
Epoch 3/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step - accuracy: 1.0000 - loss: 0.0335 - val_accuracy: 1.0000 - val_loss: 0.1076
Epoch 4/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 1.0000 - loss: 0.0276 - val_accuracy: 1.0000 - val_loss: 0.0277
Epoch 5/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step - accuracy: 1.0000 - loss: 0.0105 - val_accuracy: 1.0000 - val_loss: 0.0092


In [10]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 1.0000 - loss: 0.0057
Test Accuracy: 1.0


In [11]:
model.save("deepfake_cnn_baseline.h5")
print("Model saved!")

Model saved!


In [12]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze pretrained layers

model_tl = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_tl.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_tl.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 268s 29us/step


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [13]:
history_tl = model_tl.fit(
    X_train, y_train,
    epochs=5,
    batch_size=4,
    validation_split=0.2
)

Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 7s 658ms/step - accuracy: 1.0000 - loss: 0.2165 - val_accuracy: 1.0000 - val_loss: 0.0326
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 1.0000 - loss: 0.0082 - val_accuracy: 1.0000 - val_loss: 0.0079
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 1.0000 - loss: 0.0018 - val_accuracy: 1.0000 - val_loss: 0.0029
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 1.0000 - loss: 6.1691e-04 - val_accuracy: 1.0000 - val_loss: 0.0013
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 1.0000 - loss: 2.6310e-04 - val_accuracy: 1.0000 - val_loss: 7.3496e-04


In [14]:
loss, acc = model_tl.evaluate(X_test, y_test)
print("Transfer Learning Test Accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 1.0000 - loss: 4.8485e-04
Transfer Learning Test Accuracy: 1.0


In [15]:
import cv2
import numpy as np

def predict_video(video_path, model, frame_skip=10, img_size=224):
    cap = cv2.VideoCapture(video_path)
    preds = []

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_skip == 0:
            frame = cv2.resize(frame, (img_size, img_size))
            frame = frame / 255.0
            frame = np.expand_dims(frame, axis=0)
            pred = model.predict(frame, verbose=0)[0][0]
            preds.append(pred)

        frame_count += 1

    cap.release()

    if len(preds) == 0:
        return None

    avg_pred = np.mean(preds)
    label = "FAKE" if avg_pred > 0.5 else "REAL"
    confidence = avg_pred if label == "FAKE" else 1 - avg_pred

    return label, float(confidence)

In [16]:
print(predict_video("data/raw_videos/real/000.mp4", model_tl))
print(predict_video("data/raw_videos/fake/002_006.mp4", model_tl))

('REAL', 0.5854247212409973)
('REAL', 0.8097388744354248)


In [23]:
import shutil
import os

shutil.rmtree("data/processed_frames/real", ignore_errors=True)
shutil.rmtree("data/processed_frames/fake", ignore_errors=True)

os.makedirs("data/processed_frames/real", exist_ok=True)
os.makedirs("data/processed_frames/fake", exist_ok=True)

print("Cleared processed_frames folders")

Cleared processed_frames folders


In [26]:
print("Real frames:", len(os.listdir("data/processed_frames/real")))
print("Fake frames:", len(os.listdir("data/processed_frames/fake")))

Real frames: 0
Fake frames: 0


In [28]:
print("Real images:", len(real_images))
print("Fake images:", len(fake_images))

Real images: 0
Fake images: 0


In [29]:
import os

print("Real videos:", os.listdir("data/raw_videos/real"))
print("Fake videos:", os.listdir("data/raw_videos/fake"))

Real videos: ['000.mp4', '001.mp4', '002.mp4', '003.mp4', '004.mp4']
Fake videos: ['000_003.mp4', '001_870.mp4', '002_006.mp4', '003_000.mp4', '004_982.mp4']


In [30]:
import cv2
import os

def extract_frames_debug(video_path, out_dir, frame_skip=10):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ Could not open:", video_path)
        return

    count = 0
    saved = 0
    os.makedirs(out_dir, exist_ok=True)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % frame_skip == 0:
            out_path = os.path.join(out_dir, f"{os.path.basename(video_path)}_{saved}.jpg")
            cv2.imwrite(out_path, frame)
            saved += 1
        count += 1

    cap.release()
    print(f"✅ {video_path} → {saved} frames saved")

# Run for all videos
for v in os.listdir("data/raw_videos/real"):
    extract_frames_debug(os.path.join("data/raw_videos/real", v), "data/processed_frames/real")

for v in os.listdir("data/raw_videos/fake"):
    extract_frames_debug(os.path.join("data/raw_videos/fake", v), "data/processed_frames/fake")

✅ data/raw_videos/real\000.mp4 → 40 frames saved
✅ data/raw_videos/real\001.mp4 → 46 frames saved
✅ data/raw_videos/real\002.mp4 → 70 frames saved
✅ data/raw_videos/real\003.mp4 → 31 frames saved
✅ data/raw_videos/real\004.mp4 → 31 frames saved
✅ data/raw_videos/fake\000_003.mp4 → 31 frames saved
✅ data/raw_videos/fake\001_870.mp4 → 61 frames saved
✅ data/raw_videos/fake\002_006.mp4 → 31 frames saved
✅ data/raw_videos/fake\003_000.mp4 → 40 frames saved
✅ data/raw_videos/fake\004_982.mp4 → 85 frames saved


In [31]:
import os

print("Processed real frames:", len(os.listdir("data/processed_frames/real")))
print("Processed fake frames:", len(os.listdir("data/processed_frames/fake")))

Processed real frames: 218
Processed fake frames: 248


In [32]:
real_images, real_labels = load_images_from_folder("data/processed_frames/real", 0)
fake_images, fake_labels = load_images_from_folder("data/processed_frames/fake", 1)

print("Real images:", len(real_images))
print("Fake images:", len(fake_images))

Real images: 218
Fake images: 248


In [33]:
print("Real frames on disk:", len(os.listdir("data/processed_frames/real")))
print("Fake frames on disk:", len(os.listdir("data/processed_frames/fake")))
print("Real images in memory:", len(real_images))
print("Fake images in memory:", len(fake_images))

Real frames on disk: 218
Fake frames on disk: 248
Real images in memory: 218
Fake images in memory: 248


In [34]:
X = np.array(real_images + fake_images)
y = np.array(real_labels + fake_labels)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print("Train:", len(X_train))
print("Test:", len(X_test))


Train: 372
Test: 94


In [35]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model_tl = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_tl.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_tl.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [36]:
history_tl = model_tl.fit(
    X_train, y_train,
    epochs=5,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 7s 118ms/step - accuracy: 0.5825 - loss: 0.7081 - val_accuracy: 0.6267 - val_loss: 0.6763
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 101ms/step - accuracy: 0.6364 - loss: 0.6744 - val_accuracy: 0.4800 - val_loss: 0.8937
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step - accuracy: 0.5791 - loss: 0.7707 - val_accuracy: 0.5467 - val_loss: 0.7346
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step - accuracy: 0.6599 - loss: 0.6653 - val_accuracy: 0.5333 - val_loss: 0.7343
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step - accuracy: 0.6599 - loss: 0.6180 - val_accuracy: 0.5600 - val_loss: 0.6948


In [37]:
print(predict_video("data/raw_videos/real/000.mp4", model_tl))
print(predict_video("data/raw_videos/fake/002_006.mp4", model_tl))

('REAL', 0.5537390112876892)
('FAKE', 0.5327805876731873)


In [38]:
history_tl = model_tl.fit(
    X_train, y_train,
    epochs=15,        # increased
    batch_size=8,
    validation_split=0.2
)

Epoch 1/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - accuracy: 0.6768 - loss: 0.5966 - val_accuracy: 0.5733 - val_loss: 0.7211
Epoch 2/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - accuracy: 0.6566 - loss: 0.6530 - val_accuracy: 0.6000 - val_loss: 0.8435
Epoch 3/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 102ms/step - accuracy: 0.6768 - loss: 0.5907 - val_accuracy: 0.5600 - val_loss: 0.7183
Epoch 4/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 100ms/step - accuracy: 0.7071 - loss: 0.5577 - val_accuracy: 0.4667 - val_loss: 0.8773
Epoch 5/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 102ms/step - accuracy: 0.6835 - loss: 0.5840 - val_accuracy: 0.5333 - val_loss: 0.7489
Epoch 6/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.6902 - loss: 0.5976 - val_accuracy: 0.5467 - val_loss: 0.7685
Epoch 7/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 89ms/step - accuracy: 0.7003 - loss: 0.5741 - val_accuracy: 0.6000 - val_loss: 0.7587
Epoch 8/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - accuracy: 0.6700 - loss: 0.5948 - val_accuracy: 0.6000 

In [39]:
# Unfreeze last 20 layers of base model
for layer in base_model.layers[-20:]:
    layer.trainable = True

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # lower LR for fine-tuning
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_ft = model_tl.fit(
    X_train, y_train,
    epochs=10,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 11s 152ms/step - accuracy: 0.6498 - loss: 0.6925 - val_accuracy: 0.6000 - val_loss: 0.8664
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 115ms/step - accuracy: 0.6835 - loss: 0.5980 - val_accuracy: 0.6267 - val_loss: 0.8761
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 116ms/step - accuracy: 0.7003 - loss: 0.5860 - val_accuracy: 0.6267 - val_loss: 0.8815
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 113ms/step - accuracy: 0.7239 - loss: 0.5333 - val_accuracy: 0.6133 - val_loss: 0.8797
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 129ms/step - accuracy: 0.7643 - loss: 0.5024 - val_accuracy: 0.6133 - val_loss: 0.8840
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 130ms/step - accuracy: 0.8148 - loss: 0.4749 - val_accuracy: 0.5867 - val_loss: 0.8731
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 128ms/step - accuracy: 0.8114 - loss: 0.4326 - val_accuracy: 0.5733 - val_loss: 0.8770
Epoch 8/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 115ms/step - accuracy: 0.8013 - loss: 0.4350 - val_accuracy: 0

In [40]:
print(predict_video("data/raw_videos/real/000.mp4", model_tl))
print(predict_video("data/raw_videos/fake/002_006.mp4", model_tl))

('REAL', 0.9106702208518982)
('REAL', 0.9540629982948303)


In [41]:
print("Real frames:", len(real_images))
print("Fake frames:", len(fake_images))


Real frames: 218
Fake frames: 248


In [42]:
from sklearn.utils import class_weight
import numpy as np

class_weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {0: class_weights[0], 1: class_weights[1]}
print("Class weights:", class_weights)

Class weights: {0: np.float64(1.0449438202247192), 1: np.float64(0.9587628865979382)}


In [43]:
model_tl.fit(
    X_train, y_train,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    class_weight=class_weights
)

Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 101ms/step - accuracy: 0.8081 - loss: 0.3961 - val_accuracy: 0.5600 - val_loss: 0.8831
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.9024 - loss: 0.3282 - val_accuracy: 0.5200 - val_loss: 0.8804
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 106ms/step - accuracy: 0.9057 - loss: 0.3173 - val_accuracy: 0.5200 - val_loss: 0.8861
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 112ms/step - accuracy: 0.8822 - loss: 0.3229 - val_accuracy: 0.5200 - val_loss: 0.8976
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 119ms/step - accuracy: 0.9091 - loss: 0.2976 - val_accuracy: 0.5467 - val_loss: 0.9052
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 109ms/step - accuracy: 0.9327 - loss: 0.2637 - val_accuracy: 0.5333 - val_loss: 0.9088
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 106ms/step - accuracy: 0.9327 - loss: 0.2559 - val_accuracy: 0.5333 - val_loss: 0.9195
Epoch 8/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.9125 - loss: 0.2650 - val_accuracy: 0.

In [44]:
def predict_video_strict(video_path, model, frame_skip=10, img_size=224, fake_ratio_threshold=0.6):
    cap = cv2.VideoCapture(video_path)
    fake_votes = 0
    total = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if total % frame_skip == 0:
            frame = cv2.resize(frame, (img_size, img_size))
            frame = frame / 255.0
            frame = np.expand_dims(frame, axis=0)
            pred = model.predict(frame, verbose=0)[0][0]
            if pred > 0.5:
                fake_votes += 1
            total += 1

    cap.release()

    if total == 0:
        return None

    fake_ratio = fake_votes / total
    label = "FAKE" if fake_ratio > fake_ratio_threshold else "REAL"
    return label, fake_ratio

print(predict_video_strict("data/raw_videos/real/000.mp4", model_tl))
print(predict_video_strict("data/raw_videos/fake/002_006.mp4", model_tl))

('REAL', 0.0)
('REAL', 0.0)


In [45]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=10,
    brightness_range=[0.8,1.2],
    zoom_range=0.1
)

In [46]:
import cv2
import numpy as np

def predict_video_strict(video_path, model, frame_skip=10, img_size=224, fake_ratio_threshold=0.5):
    cap = cv2.VideoCapture(video_path)
    fake_votes = 0
    sampled = 0
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_skip == 0:
            frame_resized = cv2.resize(frame, (img_size, img_size))
            frame_norm = frame_resized / 255.0
            frame_input = np.expand_dims(frame_norm, axis=0)

            pred = model.predict(frame_input, verbose=0)[0][0]
            if pred > 0.5:
                fake_votes += 1
            sampled += 1

        frame_idx += 1

    cap.release()

    if sampled == 0:
        return "ERROR_NO_FRAMES", 0.0

    fake_ratio = fake_votes / sampled
    label = "FAKE" if fake_ratio > fake_ratio_threshold else "REAL"

    return label, fake_ratio

In [62]:
print(predict_video_strict("data/raw_videos/real/000.mp4", model_tl))
print(predict_video_strict("data/raw_videos/fake/002_006.mp4", model_tl))

('REAL', 0.075)
('REAL', 0.0)


In [48]:
print("Loose threshold (0.4):")
print(predict_video_strict("data/raw_videos/fake/002_006.mp4", model_tl, fake_ratio_threshold=0.4))

print("Strict threshold (0.6):")
print(predict_video_strict("data/raw_videos/fake/002_006.mp4", model_tl, fake_ratio_threshold=0.6))

Loose threshold (0.4):
('REAL', 0.0)
Strict threshold (0.6):
('REAL', 0.0)


In [49]:
import cv2
import os

def extract_frames_by_video(video_dir, out_root, frame_skip=5, max_frames=30):
    os.makedirs(out_root, exist_ok=True)

    for vid in os.listdir(video_dir):
        video_path = os.path.join(video_dir, vid)
        vid_name = os.path.splitext(vid)[0]
        out_dir = os.path.join(out_root, vid_name)
        os.makedirs(out_dir, exist_ok=True)

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print("❌ Could not open:", video_path)
            continue

        count = 0
        saved = 0
        frame_idx = 0

        while True:
            ret, frame = cap.read()
            if not ret or saved >= max_frames:
                break

            if frame_idx % frame_skip == 0:
                frame = cv2.resize(frame, (224,224))
                out_path = os.path.join(out_dir, f"{saved}.jpg")
                cv2.imwrite(out_path, frame)
                saved += 1

            frame_idx += 1

        cap.release()
        print(f"✅ {vid} → {saved} frames")

# Run for real and fake videos
extract_frames_by_video("data/raw_videos/real", "data/sequences/real")
extract_frames_by_video("data/raw_videos/fake", "data/sequences/fake")

✅ 000.mp4 → 30 frames
✅ 001.mp4 → 30 frames
✅ 002.mp4 → 30 frames
✅ 003.mp4 → 30 frames
✅ 004.mp4 → 30 frames
✅ 005.mp4 → 30 frames
✅ 006.mp4 → 30 frames
✅ 007.mp4 → 30 frames
✅ 008.mp4 → 30 frames
✅ 009.mp4 → 30 frames
✅ 010.mp4 → 30 frames
✅ 011.mp4 → 30 frames
✅ 012.mp4 → 30 frames
✅ 013.mp4 → 30 frames
✅ 014.mp4 → 30 frames
✅ 000_003.mp4 → 30 frames
✅ 001_870.mp4 → 30 frames
✅ 002_006.mp4 → 30 frames
✅ 003_000.mp4 → 30 frames
✅ 004_982.mp4 → 30 frames
✅ 005_010.mp4 → 30 frames
✅ 006_002.mp4 → 30 frames
✅ 007_132.mp4 → 30 frames
✅ 008_990.mp4 → 30 frames
✅ 009_027.mp4 → 30 frames
✅ 010_005.mp4 → 30 frames
✅ 011_805.mp4 → 30 frames
✅ 012_026.mp4 → 30 frames
✅ 013_883.mp4 → 30 frames
✅ 015_919.mp4 → 30 frames


In [50]:
import numpy as np
import os
import cv2

def load_sequences(root_dir, label, max_frames=30):
    X, y = [], []

    for vid_folder in os.listdir(root_dir):
        seq_path = os.path.join(root_dir, vid_folder)
        frames = []

        for img in sorted(os.listdir(seq_path)):
            img_path = os.path.join(seq_path, img)
            frame = cv2.imread(img_path)
            frame = frame / 255.0
            frames.append(frame)

        if len(frames) == max_frames:
            X.append(frames)
            y.append(label)

    return np.array(X), np.array(y)

X_real, y_real = load_sequences("data/sequences/real", 0)
X_fake, y_fake = load_sequences("data/sequences/fake", 1)

X = np.concatenate([X_real, X_fake])
y = np.concatenate([y_real, y_fake])

print("Sequences shape:", X.shape)  # (num_videos, 30, 224, 224, 3)
print("Labels shape:", y.shape)

Sequences shape: (30, 30, 224, 224, 3)
Labels shape: (30,)


In [51]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train videos:", X_train.shape[0])
print("Test videos:", X_test.shape[0])

Train videos: 24
Test videos: 6


In [52]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

cnn_base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
cnn_base.trainable = False

frame_encoder = models.Sequential([
    cnn_base,
    layers.GlobalAveragePooling2D()
])

sequence_input = layers.Input(shape=(30,224,224,3))
encoded_frames = layers.TimeDistributed(frame_encoder)(sequence_input)
lstm_out = layers.LSTM(128)(encoded_frames)
output = layers.Dense(1, activation='sigmoid')(lstm_out)

model = models.Model(sequence_input, output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 30, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 30, 1280)       │     2,257,984 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       721,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,979,521 (11.37 MB)

 Trainable params: 721,537 (2.75 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [53]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=2,
    validation_split=0.2
)

Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 90s 4s/step - accuracy: 0.3684 - loss: 1.2834 - val_accuracy: 0.0000e+00 - val_loss: 1.0271
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.6316 - loss: 0.6711 - val_accuracy: 0.0000e+00 - val_loss: 0.9665
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 16s 2s/step - accuracy: 0.7895 - loss: 0.5862 - val_accuracy: 0.0000e+00 - val_loss: 1.1915
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.7368 - loss: 0.5041 - val_accuracy: 0.0000e+00 - val_loss: 1.4195
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.7895 - loss: 0.4524 - val_accuracy: 0.0000e+00 - val_loss: 1.7769


In [54]:
cnn_base.trainable = True

for layer in cnn_base.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [55]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=2,
    validation_split=0.2
)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 143s 6s/step - accuracy: 0.6316 - loss: 0.6242 - val_accuracy: 0.0000e+00 - val_loss: 1.7497
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 37s 4s/step - accuracy: 0.8947 - loss: 0.5783 - val_accuracy: 0.0000e+00 - val_loss: 1.6865
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 0.8421 - loss: 0.5302 - val_accuracy: 0.0000e+00 - val_loss: 1.6773
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 37s 4s/step - accuracy: 0.6842 - loss: 0.5662 - val_accuracy: 0.0000e+00 - val_loss: 1.6646
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 36s 4s/step - accuracy: 0.6842 - loss: 0.6102 - val_accuracy: 0.0000e+00 - val_loss: 1.6350
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 37s 4s/step - accuracy: 0.8421 - loss: 0.5488 - val_accuracy: 0.0000e+00 - val_loss: 1.6376
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 36s 4s/step - accuracy: 0.6842 - loss: 0.5827 - val_accuracy: 0.0000e+00 - val_loss: 1.6337
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 42s 4s/step - accuracy: 0.6842 - loss: 0.5818 - val

In [56]:
def strong_video_prediction(model, video_path):
    clips = extract_multiple_sequences(video_path)  # sample 3–5 clips
    preds = [model.predict(clip[np.newaxis])[0][0] for clip in clips]

    avg_score = sum(preds) / len(preds)

    label = "FAKE" if avg_score > 0.5 else "REAL"
    return label, avg_score

In [57]:
def confidence_label(score):
    if score > 0.8 or score < 0.2:
        return "HIGH confidence"
    elif score > 0.65 or score < 0.35:
        return "MEDIUM confidence"
    else:
        return "LOW confidence"

In [58]:
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    zoom_range=0.2
)

In [64]:
"data/raw_videos/real/000.mp4"
"data/raw_videos/fake/002_006.mp4"

'data/raw_videos/fake/002_006.mp4'

In [3]:
label_real, score_real = strong_video_prediction(model, r"data/raw_videos/real/000.mp4")
print("REAL video prediction:", label_real, score_real)

label_fake, score_fake = strong_video_prediction(model, r"data/raw_videos/fake/002_006.mp4")
print("FAKE video prediction:", label_fake, score_fake)

NameError: name 'strong_video_prediction' is not defined

In [2]:
print("Loose threshold (0.6):")
print(predict_video_strict("data/raw_videos/fake/002_006.mp4", model_tl, fake_ratio_threshold=0.6))

print("Strict threshold (0.4):")
print(predict_video_strict("data/raw_videos/fake/002_006.mp4", model_tl, fake_ratio_threshold=0.4))

Loose threshold (0.6):


NameError: name 'predict_video_strict' is not defined

In [4]:
import cv2
import numpy as np
import random

def strong_video_prediction(model, video_path, threshold=0.5, samples=12):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return ("ERROR", 0.0)

    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        return ("ERROR", 0.0)

    picks = np.linspace(0, total_frames-1, samples, dtype=int)

    for idx in picks:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        frame = cv2.resize(frame, (224, 224))
        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        return ("ERROR", 0.0)

    X = np.array(frames)
    preds = model.predict(X, verbose=0)
    avg_score = float(np.mean(preds))

    label = "FAKE" if avg_score >= threshold else "REAL"
    return (label, avg_score)

In [1]:
label_real, score_real = strong_video_prediction(model, r"data/raw_videos/real/000.mp4")
print("REAL video prediction:", label_real, score_real)

label_fake, score_fake = strong_video_prediction(model, r"data/raw_videos/fake/002_006.mp4")
print("FAKE video prediction:", label_fake, score_fake)

NameError: name 'strong_video_prediction' is not defined

In [2]:
import os
import cv2
import numpy as np
import random
import tensorflow as tf

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

In [3]:
from tensorflow.keras.models import load_model

model = load_model("model/deepscan_model.h5")   # update path if different
print("✅ Model loaded")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'model/deepscan_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [4]:
import os
import cv2
import numpy as np
import random
import tensorflow as tf

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

In [18]:
os.makedirs("data/processed_frames/real", exist_ok=True)
os.makedirs("data/processed_frames/fake", exist_ok=True)

def extract_frames(video_path, out_dir, every_n=15, max_frames=50):
    cap = cv2.VideoCapture(video_path)
    count = 0
    saved = 0

    while cap.isOpened() and saved < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        if count % every_n == 0:
            frame = cv2.resize(frame, (224, 224))
            out_path = os.path.join(out_dir, f"{os.path.basename(video_path)}_{saved}.jpg")
            cv2.imwrite(out_path, frame)
            saved += 1

        count += 1

    cap.release()

# Extract REAL
for f in os.listdir("data/raw_videos/real"):
    extract_frames(os.path.join("data/raw_videos/real", f), "data/processed_frames/real")

# Extract FAKE
for f in os.listdir("data/raw_videos/fake"):
    extract_frames(os.path.join("data/raw_videos/fake", f), "data/processed_frames/fake")

print("✅ Frame extraction done")

✅ Frame extraction done


In [9]:
import cv2
import os
import numpy as np

def load_images_from_folder(folder, label, target_size=(224,224)):
    images, labels = [], []
    
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)

        if img is None:
            continue  # skip broken images

        # Force consistent size
        img = cv2.resize(img, target_size)

        # Normalize
        img = img.astype("float32") / 255.0

        images.append(img)
        labels.append(label)

    return images, labels

real_images, real_labels = load_images_from_folder("data/processed_frames/real", 0)
fake_images, fake_labels = load_images_from_folder("data/processed_frames/fake", 1)

X = np.array(real_images + fake_images, dtype="float32")
y = np.array(real_labels + fake_labels)

print("Real images:", len(real_images))
print("Fake images:", len(fake_images))
print("Total:", len(X))
print("X shape:", X.shape)

Real images: 551
Fake images: 538
Total: 1089
X shape: (1089, 224, 224, 3)


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 816
Test: 273


In [11]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')   # FAKE probability
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\Arsh\OneDrive\Desktop\DeepScan-AI\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=8,
    validation_data=(X_test, y_test)
)

Epoch 1/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 14s 126ms/step - accuracy: 0.5784 - loss: 0.6686 - val_accuracy: 0.6410 - val_loss: 0.6489
Epoch 2/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 12s 121ms/step - accuracy: 0.6422 - loss: 0.6294 - val_accuracy: 0.6374 - val_loss: 0.6209
Epoch 3/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 12s 121ms/step - accuracy: 0.6801 - loss: 0.5815 - val_accuracy: 0.6300 - val_loss: 0.6224
Epoch 4/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 12s 121ms/step - accuracy: 0.6789 - loss: 0.5619 - val_accuracy: 0.6227 - val_loss: 0.6442
Epoch 5/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 12s 120ms/step - accuracy: 0.6728 - loss: 0.5337 - val_accuracy: 0.6337 - val_loss: 0.6452
Epoch 6/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 12s 122ms/step - accuracy: 0.6961 - loss: 0.5240 - val_accuracy: 0.6447 - val_loss: 0.6117
Epoch 7/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 12s 122ms/step - accuracy: 0.6949 - loss: 0.5195 - val_accuracy: 0.6264 - val_loss: 0.6098
Epoch 8/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 12s 122ms/step - accuracy: 0.7108 - loss: 0

In [13]:
os.makedirs("model", exist_ok=True)
model.save("model/deepscan_model.h5")
print("✅ Model saved")

✅ Model saved


In [14]:
def strong_video_prediction(model, video_path, threshold=0.5, samples=12):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    picks = np.linspace(0, max(total-1, 1), samples, dtype=int)
    frames = []

    for i in picks:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.resize(frame, (224,224))
        frame = frame.astype("float32") / 255.0
        frames.append(frame)

    cap.release()

    X = np.array(frames)
    preds = model.predict(X, verbose=0).reshape(-1)
    score = float(np.mean(preds))   # FAKE confidence

    label = "FAKE" if score >= threshold else "REAL"
    return label, round(score, 4)

In [15]:
print("REAL video:", strong_video_prediction(model, r"data/raw_videos/real/000.mp4"))
print("FAKE video:", strong_video_prediction(model, r"data/raw_videos/fake/002_006.mp4"))

REAL video: ('REAL', 0.4284)
FAKE video: ('REAL', 0.4807)


In [16]:
pip install mtcnn

   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.9 MB 2.1 MB/s eta 0:00:01
   ---------------------- ----------------- 1.0/1.9 MB 2.1 MB/s eta 0:00:01
   --------------------------- ------------ 1.3/1.9 MB 1.8 MB/s eta 0:00:01
   --------------------------------- ------ 1.6/1.9 MB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 1.9/1.9 MB 1.6 MB/s  0:00:02

   ---------------------------------------- 2/2 [mtcnn]

Note: you may

In [1]:
import os

os.makedirs("data/processed_frames_face/real", exist_ok=True)
os.makedirs("data/processed_frames_face/fake", exist_ok=True)

print("Folders ready")

Folders ready


In [2]:
import cv2
from mtcnn import MTCNN

detector = MTCNN()

def extract_face_frames(video_path, out_dir, every_n=15, max_faces=30):
    cap = cv2.VideoCapture(video_path)
    count = 0
    saved = 0

    if not cap.isOpened():
        print("❌ Cannot open:", video_path)
        return

    while cap.isOpened() and saved < max_faces:
        ret, frame = cap.read()
        if not ret:
            break

        if count % every_n == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            faces = detector.detect_faces(rgb)

            if len(faces) > 0:
                x, y, w, h = faces[0]['box']

                # Fix negative coords (MTCNN bug sometimes)
                x, y = max(0, x), max(0, y)
                face = frame[y:y+h, x:x+w]

                if face.size != 0:
                    face = cv2.resize(face, (224, 224))
                    out_path = os.path.join(out_dir, f"{os.path.basename(video_path)}_{saved}.jpg")
                    cv2.imwrite(out_path, face)
                    saved += 1

        count += 1

    cap.release()
    print(f"✅ Extracted {saved} faces from {os.path.basename(video_path)}")

In [3]:
# REAL videos
for f in os.listdir("data/raw_videos/real"):
    extract_face_frames(
        os.path.join("data/raw_videos/real", f),
        "data/processed_frames_face/real"
    )

# FAKE videos
for f in os.listdir("data/raw_videos/fake"):
    extract_face_frames(
        os.path.join("data/raw_videos/fake", f),
        "data/processed_frames_face/fake"
    )

✅ Extracted 27 faces from 000.mp4
✅ Extracted 30 faces from 001.mp4
✅ Extracted 30 faces from 002.mp4
✅ Extracted 21 faces from 003.mp4
✅ Extracted 21 faces from 004.mp4
✅ Extracted 26 faces from 005.mp4
✅ Extracted 21 faces from 006.mp4
✅ Extracted 30 faces from 007.mp4
✅ Extracted 30 faces from 008.mp4
✅ Extracted 30 faces from 009.mp4
✅ Extracted 30 faces from 010.mp4
✅ Extracted 30 faces from 011.mp4
✅ Extracted 25 faces from 012.mp4
✅ Extracted 23 faces from 013.mp4
✅ Extracted 30 faces from 014.mp4
✅ Extracted 21 faces from 000_003.mp4
✅ Extracted 30 faces from 001_870.mp4
✅ Extracted 21 faces from 002_006.mp4
✅ Extracted 27 faces from 003_000.mp4
✅ Extracted 30 faces from 004_982.mp4
✅ Extracted 30 faces from 005_010.mp4
✅ Extracted 30 faces from 006_002.mp4
✅ Extracted 28 faces from 007_132.mp4
✅ Extracted 21 faces from 008_990.mp4
✅ Extracted 21 faces from 009_027.mp4
✅ Extracted 26 faces from 010_005.mp4
✅ Extracted 28 faces from 011_805.mp4
✅ Extracted 22 faces from 012_026.

In [4]:
print("Real face images:", len(os.listdir("data/processed_frames_face/real")))
print("Fake face images:", len(os.listdir("data/processed_frames_face/fake")))

Real face images: 404
Fake face images: 394


In [5]:
import os, cv2, numpy as np

def load_images_from_folder(folder, label, target_size=(224,224)):
    images, labels = [], []
    for f in os.listdir(folder):
        p = os.path.join(folder, f)
        img = cv2.imread(p)
        if img is None:
            continue
        img = cv2.resize(img, target_size)
        img = img.astype("float32") / 255.0
        images.append(img)
        labels.append(label)
    return images, labels

real_images, real_labels = load_images_from_folder("data/processed_frames_face/real", 0)
fake_images, fake_labels = load_images_from_folder("data/processed_frames_face/fake", 1)

X = np.array(real_images + fake_images, dtype="float32")
y = np.array(real_labels + fake_labels)

print("Real faces:", len(real_images))
print("Fake faces:", len(fake_images))
print("X shape:", X.shape)

Real faces: 404
Fake faces: 394
X shape: (798, 224, 224, 3)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 598
Test: 200


In [7]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base.trainable = False  # freeze backbone

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")  # FAKE probability
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=16,
    validation_data=(X_test, y_test)
)

Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 12s 227ms/step - accuracy: 0.5401 - loss: 0.7670 - val_accuracy: 0.7400 - val_loss: 0.5829
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 7s 185ms/step - accuracy: 0.7124 - loss: 0.5555 - val_accuracy: 0.8350 - val_loss: 0.4871
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 7s 177ms/step - accuracy: 0.7759 - loss: 0.4722 - val_accuracy: 0.7900 - val_loss: 0.4456
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 162ms/step - accuracy: 0.7709 - loss: 0.4545 - val_accuracy: 0.8650 - val_loss: 0.3855
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 163ms/step - accuracy: 0.8361 - loss: 0.3656 - val_accuracy: 0.8550 - val_loss: 0.3594
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 160ms/step - accuracy: 0.8796 - loss: 0.3165 - val_accuracy: 0.8750 - val_loss: 0.3379
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 165ms/step - accuracy: 0.8863 - loss: 0.2927 - val_accuracy: 0.8200 - val_loss: 0.3585
Epoch 8/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 169ms/step - accuracy: 0.8963 - loss: 0.2828 - val_accuracy: 0

In [9]:
import os
os.makedirs("model", exist_ok=True)
model.save("model/deepscan_face_mobilenetv2.h5")
print("✅ Model saved")

✅ Model saved


In [10]:
import numpy as np, cv2
from mtcnn import MTCNN

detector = MTCNN()

def strong_video_prediction_face(model, video_path, threshold=0.5, samples=24):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    picks = np.linspace(0, max(total-1, 1), samples, dtype=int)

    faces = []
    for i in picks:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        dets = detector.detect_faces(rgb)
        if len(dets) == 0:
            continue

        x, y, w, h = dets[0]['box']
        x, y = max(0, x), max(0, y)
        face = frame[y:y+h, x:x+w]
        if face.size == 0:
            continue

        face = cv2.resize(face, (224,224))
        face = face.astype("float32") / 255.0
        faces.append(face)

    cap.release()

    if len(faces) == 0:
        return ("ERROR: No face detected", 0.0)

    preds = model.predict(np.array(faces), verbose=0).reshape(-1)
    score = float(np.mean(preds))  # FAKE confidence

    label = "FAKE" if score >= threshold else "REAL"
    return label, round(score, 4)

In [11]:
print("REAL video:", strong_video_prediction_face(model, r"data/raw_videos/real/000.mp4"))
print("FAKE video:", strong_video_prediction_face(model, r"data/raw_videos/fake/002_006.mp4"))

REAL video: ('REAL', 0.1063)
FAKE video: ('FAKE', 0.79)


In [12]:
def classify_with_confidence(label, score):
    if score < 0.4:
        return "REAL (High confidence)", score
    elif score > 0.6:
        return "FAKE (High confidence)", score
    else:
        return "UNCERTAIN (Low confidence)", score

In [13]:
label, score = strong_video_prediction_face(model, r"data/raw_videos/real/000.mp4")
print(classify_with_confidence(label, score))

('REAL (High confidence)', 0.1063)


In [14]:
model.save("model/deepscan_face_mobilenetv2.h5")

In [15]:
from tensorflow.keras.models import load_model
model = load_model("model/deepscan_face_mobilenetv2.h5")

In [1]:
import shutil, os

for folder in ["data/processed_frames/real", "data/processed_frames/fake"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

print("Old processed frames cleared.")

PermissionError: [WinError 5] Access is denied: 'data/processed_frames/real'

In [7]:
import os, shutil

for folder in ["data/processed_frames_faces/real", "data/processed_frames_faces/fake"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

print("Clean folders ready.")

PermissionError: [WinError 5] Access is denied: 'data/processed_frames_faces/real'

In [8]:
import cv2, os
from mtcnn.mtcnn import MTCNN

detector = MTCNN()

IMG_SIZE = 224   # keep same for training stability
FRAME_SKIP = 10  # already good
MAX_FACES = 40   # limit per video

def extract_faces(video_path, out_dir, max_faces=40, frame_skip=10):
    cap = cv2.VideoCapture(video_path)
    saved = 0
    frame_id = 0

    while cap.isOpened() and saved < max_faces:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % frame_skip == 0:
            faces = detector.detect_faces(frame)
            if faces:
                x, y, w, h = faces[0]['box']
                x, y = max(0, x), max(0, y)
                face = frame[y:y+h, x:x+w]
                if face.size > 0:
                    face = cv2.resize(face, (224,224))
                    name = f"{os.path.splitext(os.path.basename(video_path))[0]}_{saved}.jpg"
                    cv2.imwrite(os.path.join(out_dir, name), face)
                    saved += 1

        frame_id += 1

    cap.release()

# REAL videos
for v in os.listdir("data/raw_videos/real"):
    extract_faces(f"data/raw_videos/real/{v}", "data/processed_frames_faces/real")

# FAKE videos
for v in os.listdir("data/raw_videos/fake"):
    extract_faces(f"data/raw_videos/fake/{v}", "data/processed_frames_faces/fake")

print("Face extraction complete.")

Face extraction complete.


In [9]:
import os, cv2
import numpy as np

def load_images(folder, label):
    X, y = [], []
    for f in os.listdir(folder):
        p = os.path.join(folder, f)
        img = cv2.imread(p)
        if img is not None:
            img = cv2.resize(img, (224,224))
            X.append(img)
            y.append(label)
    return X, y

real_images, real_labels = load_images("data/processed_frames_faces/real", 0)
fake_images, fake_labels = load_images("data/processed_frames_faces/fake", 1)

X = np.array(real_images + fake_images, dtype=np.float32) / 255.0
y = np.array(real_labels + fake_labels)

print("Real face images:", len(real_images))
print("Fake face images:", len(fake_images))
print("Total samples:", len(X))

Real face images: 1096
Fake face images: 1109
Total samples: 2205


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 1764
Test: 441


In [11]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze backbone for stability

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_train, y_train, epochs=6, validation_data=(X_test, y_test))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/6
56/56 ━━━━━━━━━━━━━━━━━━━━ 20s 313ms/step - accuracy: 0.6633 - loss: 0.6220 - val_accuracy: 0.7914 - val_loss: 0.4937
Epoch 2/6
56/56 ━━━━━━━━━━━━━━━━━━━━ 17s 300ms/step - accuracy: 0.7880 - loss: 0.4606 - val_accuracy: 0.8549 - val_loss: 0.3916
Epoch 3/6
56/56 ━━━━━━━━━━━━━━━━━━━━ 16s 280ms/step - accuracy: 0.8390 - loss: 0.3785 - val_accuracy: 0.8481 - val_loss: 0.3808
Epoch 4/6
56/56 ━━━━━━━━━━━━━━━━━━━━ 16s 279ms/step - accuracy: 0.8668 - loss: 0.3247 - val_accuracy: 0.8776 - val_loss: 0.3416
Epoch 5/6
56/56 ━━━━━━━━━━━━━━━━━━━━ 16s 279ms/step - accuracy: 0.8912 - loss: 0.2654 - val_accuracy: 0.8889 - val_loss: 0.3062
Epoch 6/6
56/56 ━━━━━━━━━━━━━━━━━━━━ 17s 299ms/step - accuracy: 0.9031 - loss: 0.2370 - val_accuracy: 0.8798 - val_loss: 0.2791


In [7]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [12]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=16
)

Epoch 1/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 18s 158ms/step - accuracy: 0.8946 - loss: 0.2432 - val_accuracy: 0.8798 - val_loss: 0.3169
Epoch 2/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 15s 131ms/step - accuracy: 0.9155 - loss: 0.2095 - val_accuracy: 0.8707 - val_loss: 0.2828
Epoch 3/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 15s 132ms/step - accuracy: 0.9303 - loss: 0.1833 - val_accuracy: 0.8912 - val_loss: 0.2358
Epoch 4/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 17s 153ms/step - accuracy: 0.9303 - loss: 0.1762 - val_accuracy: 0.9116 - val_loss: 0.2468
Epoch 5/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 17s 156ms/step - accuracy: 0.9456 - loss: 0.1386 - val_accuracy: 0.8957 - val_loss: 0.2632
Epoch 6/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 17s 151ms/step - accuracy: 0.9507 - loss: 0.1202 - val_accuracy: 0.9002 - val_loss: 0.2754
Epoch 7/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 18s 159ms/step - accuracy: 0.9677 - loss: 0.1011 - val_accuracy: 0.8889 - val_loss: 0.3157
Epoch 8/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 17s 156ms/step - accuracy: 0.9688 - loss: 0

In [13]:
# Save trained model
model.save("trustlens_model_v1.h5")

print("Model saved as trustlens_model_v1.h5")

Model saved as trustlens_model_v1.h5


In [14]:
import numpy as np
import cv2
from mtcnn import MTCNN

detector = MTCNN()

# Safe speed config
PRED_IMG_SIZE = 224
FRAME_SKIP = 12     # analyze fewer frames
MAX_FACES = 40      # cap samples per video
THRESHOLD = 0.6     # stricter = stronger predictions

def strong_video_prediction(model, video_path):
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        return ("ERROR", 0.0)

    faces = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        if frame_count % FRAME_SKIP != 0:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        detections = detector.detect_faces(rgb)

        for face in detections:
            x, y, w, h = face['box']
            x, y = max(0, x), max(0, y)

            face_img = rgb[y:y+h, x:x+w]
            if face_img.size == 0:
                continue

            face_img = cv2.resize(face_img, (PRED_IMG_SIZE, PRED_IMG_SIZE))
            face_img = face_img / 255.0
            faces.append(face_img)

            if len(faces) >= MAX_FACES:
                break

        if len(faces) >= MAX_FACES:
            break

    cap.release()

    if len(faces) == 0:
        return ("NO FACE DETECTED", 0.0)

    faces = np.array(faces)
    
    preds = model.predict(faces, verbose=0)
    avg_score = float(np.mean(preds))

    label = "FAKE" if avg_score > THRESHOLD else "REAL"
    confidence = avg_score if label == "FAKE" else (1 - avg_score)

    return (label, round(confidence, 4))

In [15]:
print("REAL video:", strong_video_prediction(model, "data/raw_videos/real/026.mp4"))
print("FAKE video:", strong_video_prediction(model, "data/raw_videos/fake/026_012.mp4"))

REAL video: ('REAL', 0.8579)
FAKE video: ('FAKE', 0.633)


In [17]:
print("REAL video:", strong_video_prediction(model, "data/raw_videos/real/026.mp4"))
print("FAKE video:", strong_video_prediction(model, "data/raw_videos/fake/025_067.mp4"))

REAL video: ('REAL', 0.8579)
FAKE video: ('FAKE', 0.8922)


In [19]:
import os, shutil

for folder in ["data/processed_frames_faces/real", "data/processed_frames_faces/fake"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

print("Clean folders ready.")

PermissionError: [WinError 5] Access is denied: 'data/processed_frames_faces/real'

In [20]:
import cv2, os
from mtcnn.mtcnn import MTCNN

detector = MTCNN()

def extract_faces(video_path, out_dir, max_faces=30, frame_skip=12):
    cap = cv2.VideoCapture(video_path)
    saved = 0
    frame_id = 0

    while cap.isOpened() and saved < max_faces:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % frame_skip == 0:
            faces = detector.detect_faces(frame)
            if faces:
                x, y, w, h = faces[0]['box']
                x, y = max(0, x), max(0, y)
                face = frame[y:y+h, x:x+w]
                if face.size > 0:
                    face = cv2.resize(face, (224,224))
                    name = f"{os.path.splitext(os.path.basename(video_path))[0]}_{saved}.jpg"
                    cv2.imwrite(os.path.join(out_dir, name), face)
                    saved += 1

        frame_id += 1

    cap.release()

for v in os.listdir("data/raw_videos/real"):
    extract_faces(f"data/raw_videos/real/{v}", "data/processed_frames_faces/real")

for v in os.listdir("data/raw_videos/fake"):
    extract_faces(f"data/raw_videos/fake/{v}", "data/processed_frames_faces/fake")

print("Face extraction complete.")

Face extraction complete.


In [21]:
import os, cv2
import numpy as np

def load_images(folder, label):
    X, y = [], []
    for f in os.listdir(folder):
        p = os.path.join(folder, f)
        img = cv2.imread(p)
        if img is not None:
            img = cv2.resize(img, (224,224))
            X.append(img)
            y.append(label)
    return X, y

real_images, real_labels = load_images("data/processed_frames_faces/real", 0)
fake_images, fake_labels = load_images("data/processed_frames_faces/fake", 1)

X = np.array(real_images + fake_images, dtype=np.float32) / 255.0
y = np.array(real_labels + fake_labels)

print("Real face images:", len(real_images))
print("Fake face images:", len(fake_images))
print("Total samples:", len(X))

Real face images: 846
Fake face images: 854
Total samples: 1700


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 1360
Test: 340


In [23]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))

for layer in base_model.layers[:-30]:
    layer.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True
)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=16),
    validation_data=(X_test, y_test),
    epochs=10
)

model.summary()

Epoch 1/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 23s 215ms/step - accuracy: 0.6801 - loss: 0.5888 - val_accuracy: 0.5118 - val_loss: 1.1815
Epoch 2/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 17s 195ms/step - accuracy: 0.8478 - loss: 0.3300 - val_accuracy: 0.5353 - val_loss: 1.4201
Epoch 3/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 17s 194ms/step - accuracy: 0.9213 - loss: 0.2191 - val_accuracy: 0.5324 - val_loss: 1.6715
Epoch 4/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 19s 226ms/step - accuracy: 0.9206 - loss: 0.1912 - val_accuracy: 0.5559 - val_loss: 1.5335
Epoch 5/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 251ms/step - accuracy: 0.9419 - loss: 0.1482 - val_accuracy: 0.6265 - val_loss: 1.1842
Epoch 6/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 21s 249ms/step - accuracy: 0.9471 - loss: 0.1377 - val_accuracy: 0.5471 - val_loss: 2.0417
Epoch 7/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 21s 248ms/step - accuracy: 0.9618 - loss: 0.0948 - val_accuracy: 0.6000 - val_loss: 1.5556
Epoch 8/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 21s 251ms/step - accuracy: 0.9676 - loss: 0.0892 - val_accu

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,803,077 (22.14 MB)

 Trainable params: 1,690,497 (6.45 MB)

 Non-trainable params: 731,584 (2.79 MB)

 Optimizer params: 3,380,996 (12.90 MB)

In [24]:
model.save("trustlens_deepfake_v1.h5")
print("Model saved as trustlens_deepfake_v1.h5")

Model saved as trustlens_deepfake_v1.h5


In [25]:
import numpy as np
import cv2
from mtcnn.mtcnn import MTCNN

detector = MTCNN()

def strong_video_prediction(model, video_path, threshold=0.5, frame_skip=15):
    cap = cv2.VideoCapture(video_path)
    preds = []
    frame_id = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % frame_skip == 0:
            faces = detector.detect_faces(frame)
            for face in faces:
                x, y, w, h = face['box']
                x, y = max(0, x), max(0, y)
                face_img = frame[y:y+h, x:x+w]
                if face_img.size == 0:
                    continue

                face_img = cv2.resize(face_img, (224,224))
                face_img = face_img / 255.0
                face_img = np.expand_dims(face_img, axis=0)

                pred = model.predict(face_img, verbose=0)[0][0]
                preds.append(pred)

        frame_id += 1

    cap.release()

    if len(preds) == 0:
        return ("NO FACE", 0.0)

    avg_score = float(np.mean(preds))
    label = "FAKE" if avg_score > threshold else "REAL"

    return (label, round(avg_score, 4))

In [26]:
print("REAL video:", strong_video_prediction(model, "data/raw_videos/real/026.mp4"))
print("FAKE video:", strong_video_prediction(model, "data/raw_videos/fake/026_012.mp4"))

REAL video: ('REAL', 0.1696)
FAKE video: ('FAKE', 0.9897)
